# **configuring folders**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# **SPECIFY MODEL NAME EVERY RUN!!!**

In [ ]:
# specify dataset path once at the beginning of the script
import os
import sys

#flag for checking if savepath for model was created correctly
MODEL_SAVEPATH_SAFE= False

DATASET_PATH = '/content/drive/MyDrive/THESIS/dataset-and-notebook/from-kaggle-dataset'

# modify accordingly (if needed)
NORMAL_FOLDER_PATH = f'{DATASET_PATH}/Normal'
TUBERCULOSIS_FOLDER_PATH = f'{DATASET_PATH}/Tuberculosis'

NORMAL_CSV_PATH = f'{DATASET_PATH}/Normal.metadata.xlsx'
TUBERCULOSIS_CSV_PATH = f'{DATASET_PATH}/Tuberculosis.metadata.xlsx'

# SEGMENTED DATASET TO TRAIN USING SEGMENTED IMAGEZ
SEGMENTED_DATASET = "/content/drive/MyDrive/THESIS/dataset-and-notebook/SEGMENTED-IMAGES"

# specify version at the beginning of the script, only need to change the model name, (savepath if not configured properly)
MODEL_NAME = 'resnet50_V0_6_3'
MODEL_SAVEPATH = f'/content/drive/MyDrive/THESIS/dataset-and-notebook/notebooks/ResNet50/resnet50_V0_6_/{MODEL_NAME}'
#create directory just in case
try:
    if os.path.exists(MODEL_SAVEPATH):
        print("Model might already be existing! Please change model name")
        sys.exit()
    else:
        os.makedirs(MODEL_SAVEPATH)
        print(f"File with filename {MODEL_NAME} succesfully created in {MODEL_SAVEPATH}")
        MODEL_SAVEPATH_SAFE= True
except SystemExit:
    print("Execution stopped. Session flagged for possible inconsistencies in saving model")

print('directories set succesfully')

# **DEPENDENCIES**

In [ ]:
import os
import time
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import shutil
from sklearn.model_selection import train_test_split
import csv

from tensorflow.keras.applications.resnet50 import ResNet50

#image data generators
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import preprocess_input

#model itself
from tensorflow.keras import layers,models
#from keras.applications import ResNet50
from keras.models import Sequential
from keras.layers import Dense, Activation, Flatten, Dropout, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras.optimizers import Adam # - Works
from keras.callbacks import CSVLogger, ReduceLROnPlateau, ModelCheckpoint, EarlyStopping

#supposed dapat pakita diagram ng layers
import pydot
from tensorflow.keras.utils import plot_model
from PIL import Image
import PIL

#for testing using only a single image
from keras.preprocessing.image import load_img, img_to_array

#for evaluation confusion amtrix etc.
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, roc_curve, roc_auc_score
from sklearn.preprocessing import LabelEncoder # for encoding 0 or 1 in normal or t

#for grad cam
from tensorflow.keras.models import Model
import cv2
from tensorflow.keras.preprocessing import image
from IPython.display import Image, display
import matplotlib as mpl

#for loading model from saved file
from keras.models import load_model

#plotting training statistics
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.style  as style

print ("Loaded dependencies")

import keras

print("TensorFlow version:", tf.__version__)
print("Keras version:", keras.__version__)
from tensorflow.keras.layers.experimental import preprocessing


# **HYperparameters**

In [ ]:
# modify accordingly (if needed, this should automatically be adjusted depending on the above code for model name/savepath)
# BE CAREFUL, if ntbk is run again it will overwrite all files if the model name is not changed
MODEL_PLOT = f'{MODEL_SAVEPATH}/model_plot.png'
MODEL_CSV_LOGGER = f'{MODEL_SAVEPATH}/training_log_{MODEL_NAME}.csv'
MODEL_CHECKPOINT = f'{MODEL_SAVEPATH}/{MODEL_NAME}.keras'


# good examples
tb = f'{TUBERCULOSIS_FOLDER_PATH}/Tuberculosis-10.png'
normal = f'{NORMAL_FOLDER_PATH}/Normal-3000.png'

# high score confidence achieved using efficientnet b3 v0-5
tb_highscore_efficientnetb3_v05 = f'{TUBERCULOSIS_FOLDER_PATH}/Tuberculosis-284.png'
# predicted as normal but tb positive, and also has good heatmap
tb_falsepositive = f'{TUBERCULOSIS_FOLDER_PATH}/Tuberculosis-136.png'

# bad examples
tb_bad = f'{TUBERCULOSIS_FOLDER_PATH}/Tuberculosis-330.png'
normal_bad = f'{NORMAL_FOLDER_PATH}/Normal-3500.png'
weird_image = f'{NORMAL_FOLDER_PATH}/Normal-750.png'

#-------------Editable parameters below------------------------------#

# distribution ng images
TRAIN_SPLIT = 0.85
VALIDATION_SPLIT = 0.05
TEST_SPLIT = 0.10

# training parameters:
#Making custom flow from dataframe from the generators
BATCH_SIZE = 8
IMAGE_HEIGHT = 512
IMAGE_WIDTH = 512
EPOCHS = 10
ADAM = 0.0002

print("parameters initialized")

# **splitting data (Non SEGMENTED IMAGES, FROM KAGGLE**

In [ ]:
main_folder = DATASET_PATH

normal_folder = NORMAL_FOLDER_PATH
tuberculosis_folder = TUBERCULOSIS_FOLDER_PATH

normal_df = pd.read_excel(NORMAL_CSV_PATH)
tuberculosis_df = pd.read_excel(TUBERCULOSIS_CSV_PATH)

# para 700 lang sa normal
normal_df_sampled = normal_df.sample(n=700, random_state=1)

# Add 'FILE PATH' column to each DataFrame
for index, row in normal_df_sampled.iterrows():
    normal_df_sampled.loc[index, 'FILE PATH'] = os.path.join(normal_folder, row['FILE NAME'] + '.png')

for index, row in tuberculosis_df.iterrows():
    tuberculosis_df.loc[index, 'FILE PATH'] = os.path.join(tuberculosis_folder, row['FILE NAME'] + '.png')

#print(normal_df['FILE PATH'].head())
#print(tuberculosis_df['FILE PATH'].head())


# splitting the data
# combine normal and tuberculosis DataFrames
combined_df = pd.concat([normal_df_sampled, tuberculosis_df], ignore_index=True)

# Shuffle the combined DataFrame
shuffled_df = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Add labels based on folder names
shuffled_df['LABEL'] = shuffled_df['FILE PATH'].apply(lambda x: 'normal' if 'Normal' in x else 'tuberculosis')


#num_normal_samples = (shuffled_df['label'] == 'normal').sum()

# Split the data into features (X) and target variable (y)
X = shuffled_df.drop(columns=['LABEL'])  # Features
y = shuffled_df['LABEL']                 # Target variable

# Split the data into train and temp sets based on features and target variable
X_train, X_temp, y_train, y_temp = train_test_split(X, y, train_size=TRAIN_SPLIT, random_state=42)

# Further split the temp_data into test and validation sets
X_test, X_val, y_test, y_val = train_test_split(X_temp, y_temp, test_size=VALIDATION_SPLIT/(TEST_SPLIT + VALIDATION_SPLIT), random_state=42)

# Create train, test, and validation DataFrames
train_df = X_train.copy()
train_df['LABEL'] = y_train
test_df = X_test.copy()
test_df['LABEL'] = y_test
val_df = X_val.copy()
val_df['LABEL'] = y_val

# Print class distribution in each set
print("Train set class distribution:\n", train_df['LABEL'].value_counts())
print("\nTest set class distribution:\n", test_df['LABEL'].value_counts())
print("\nValidation set class distribution:\n", val_df['LABEL'].value_counts())

#**Splitting dataset for segmented images (Shenzhen and Montgom)**

In [ ]:
# FOR SEGMENTED IMAGES WHICH CONTAINS ROUGHLY 300 TB and 300 NORMAL IMAGES
data=[]

# Iterate through the subdirectories
for subdir in ['Normal', 'Tuberculosis']:
    subdir_path = os.path.join(SEGMENTED_DATASET, subdir)

    # Iterate through the files in the subdirectory
    for filename in os.listdir(subdir_path):
        # Check the file extension
        if filename.endswith('.png'):
            # Get the file path
            file_path = os.path.join(subdir_path, filename)

            # Determine the label based on the file name
            if filename.endswith('_0.png'):
                label = 'Non-TB'
            elif filename.endswith('_1.png'):
                label = 'Tuberculosis'
            else:
                label = 'Unknown'

            # Add the data to the list
            data.append({
                'FILE PATH': file_path,
                'FILE NAME': filename,
                'LABEL': label
            })

# Create the dataframe
df_segmented = pd.DataFrame(data)

# Split the data into features (X) and target variable (y)
X_seg = df_segmented.drop(columns=['LABEL'])  # Features
y_seg = df_segmented['LABEL']                 # Target variable

# Split the data into train and temp sets based on features and target variable
X_train_S, X_temp_S, y_train_S, y_temp_S = train_test_split(X_seg, y_seg, train_size = TRAIN_SPLIT, random_state=69)

# Further split the temp_data into test and validation sets
X_test_S, X_val_S, y_test_S, y_val_S = train_test_split(X_temp_S, y_temp_S, test_size = VALIDATION_SPLIT/(TEST_SPLIT + VALIDATION_SPLIT), random_state=69)

# Create train, test, and validation DataFrames
train_df_seg = X_train_S.copy()
train_df_seg['LABEL'] = y_train_S
test_df_seg = X_test_S.copy()
test_df_seg['LABEL'] = y_test_S
val_df_seg = X_val_S.copy()
val_df_seg['LABEL'] = y_val_S

# Print class distribution in each set
print("Train set class distribution:\n", train_df_seg['LABEL'].value_counts())
print("\nValidation set class distribution:\n", val_df_seg['LABEL'].value_counts())
print("\nTest set class distribution:\n", test_df_seg['LABEL'].value_counts())

# **data generators NOTE: MAKE SURE DATAFRAMES LOADED IN FLOW FROM DATAFRAME ARE OF THE SEGMENTED IMAGES (if training using segmented images(**

In [ ]:
# Create training data generator
train_generator = ImageDataGenerator(
    preprocessing_function = preprocess_input,
    zoom_range=0.1,
    rotation_range=10,
    brightness_range=[0.9, 1.1],
    horizontal_flip=True,
    fill_mode='nearest'

    )

# Create validating data generator
val_generator = ImageDataGenerator(
    preprocessing_function = preprocess_input
    )

# Create testing data generator
test_generator  = ImageDataGenerator(
    preprocessing_function = preprocess_input
    )


# Flow training images in batches from the training DataFrame
trainResnet = train_generator.flow_from_dataframe(
    dataframe=train_df_seg,
    x_col='FILE PATH',  # column containing the file paths of the images
    y_col='LABEL',      # column containing the labels
    target_size=(IMAGE_HEIGHT, IMAGE_WIDTH),  # target size of the images
    batch_size=BATCH_SIZE,
    class_mode='binary',  # for binary classification
    shuffle=True,         # shuffle the data
    seed=42              # optional random seed for reproducibility
)

# Flow validation images in batches from the validation DataFrame
valResnet = val_generator.flow_from_dataframe(
    dataframe=val_df_seg,
    x_col='FILE PATH',
    y_col='LABEL',
    target_size=(IMAGE_HEIGHT, IMAGE_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False,
)

# Flow test images in batches from the test DataFrame
testResnet = test_generator.flow_from_dataframe(
    dataframe=test_df_seg,
    x_col='FILE PATH',
    y_col='LABEL',
    target_size=(IMAGE_HEIGHT, IMAGE_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False,  # No need to shuffle the test data
)

class_indices_trainResnet = trainResnet.class_indices
class_indices_valResnet = valResnet.class_indices
class_indices_testResnet = testResnet.class_indices


print("Class indices for train:", class_indices_trainResnet)
print("Class indices for val:", class_indices_valResnet)
print("Class indices for test:", class_indices_testResnet)
#print(test.classes)

# **Creating the model

In [ ]:
#adapted from https://www.kaggle.com/code/vencerlanz09/sea-animals-classification-using-efficeintnetb7/notebook#%F0%9F%A4%B9Training-the-model

# Data Augmentation Step (NO LONGER USED!!!)
augment = tf.keras.Sequential([
  layers.experimental.preprocessing.Resizing(IMAGE_HEIGHT, IMAGE_WIDTH),
 # layers.experimental.preprocessing.Rescaling(1./255),
  layers.experimental.preprocessing.RandomFlip("horizontal"),
  layers.experimental.preprocessing.RandomRotation(0.1),
  layers.experimental.preprocessing.RandomZoom(0.1),
  layers.experimental.preprocessing.RandomContrast(0.1),
])

base_model = ResNet50(
    input_shape=(IMAGE_HEIGHT, IMAGE_WIDTH, 3),
    include_top= False,
    weights= 'imagenet',
    pooling='max'
)

#base_model.load_weights('/kaggle/input/efficientnetb7-weights/efficientnetb7_notop.h5')
base_model.trainable = False

inputs = base_model.input

#x = augment(inputs) # NO LONGER USING THE CUSTOM LAYER
x = Dense(128, activation='relu')(base_model.output)
x = Dropout(0.3)(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.3)(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.3)(x)


outputs = Dense(1, activation='sigmoid')(x)

model_ResNet50 = Model(inputs=inputs, outputs=outputs, name= MODEL_NAME)

model_ResNet50.compile(
    optimizer=Adam(ADAM), #change accordingly
    loss='binary_crossentropy',
    metrics=['accuracy', "AUC"]
)

model_ResNet50.summary()

# Plot the model architecture
plot_model(model_ResNet50, to_file=MODEL_PLOT, show_shapes=True, show_layer_names=True)
# Load and display the image
img = PIL.Image.open(MODEL_PLOT)
plt.imshow(img)
plt.axis('off')  # Hide axis
plt.show()

# **training the model and configurations**

In [ ]:
# Define the ReduceLROnPlateau callback
reduce_lr = ReduceLROnPlateau(monitor='val_loss',   # Metric to monitor
                              factor=0.5,          # Factor by which the learning rate will be reduced
                              patience=2,          # Number of epochs with no improvement after which learning rate will be reduced
                              min_lr=1e-8)         # Lower bound on the learning rate


#please change name for every training session, pati checkpoint
csv_logger = CSVLogger(MODEL_CSV_LOGGER)

checkpoint_path = MODEL_CHECKPOINT
checkpoint_dir = os.path.dirname(checkpoint_path)

model_checkpoint = ModelCheckpoint(filepath= checkpoint_path, verbose =1, save_best_only=True, monitor="val_loss",  mode="min",)

early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose =1)

# Recording time
start = time.time()

historyResNet50 = model_ResNet50.fit(
    trainResnet,
    validation_data=valResnet,
    epochs=20,
    callbacks=[
        early_stopping,
        model_checkpoint,
        reduce_lr,
        csv_logger
    ]
)

stop = time.time()
print(f"Training time: {stop - start}s")
with open(MODEL_CSV_LOGGER, mode='a', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['Training Duration (in seconds): ', stop - start])

In [ ]:
# Load the Keras model in SavedModel format
model = load_model('/content/drive/MyDrive/THESIS/dataset-and-notebook/notebooks/efficientNetB3/efficientNetB3_V0_7/efficientnetB3_V0_7_11/efficientnetB3_V0_7_11.keras')

# Save the model in HDF5 format
model.save('/content/drive/MyDrive/THESIS/dataset-and-notebook/notebooks/efficientNetB3/efficientNetB3_V0_7/efficientnetB3_V0_7_11/efficientnetB3_V0_7_11.h5')

# **Plotting functions and also evaluating/predict functions**

In [ ]:
#MAKE SURE NA INITIALIZED ANG GENERATORS FIRST

def plot_training_hist(keras_model):
    # Plot training History
    #Creates two plots of model training logs

    hist = keras_model.history
    style.use('fivethirtyeight')

    # Loss Plot
    fig = plt.figure(figsize=(12,4))
    plt.title('Loss Plot')
    plt.plot(hist['loss'], '#07e9ed', marker='o')
    plt.plot(hist['val_loss'], '#0791ed',  marker='o')
    plt.ylim([0,1.2 * max(max(hist['loss'], hist['val_loss']))])

    plt.xlabel('Epoch')  # Add label to the x-axis
    plt.ylabel('Loss')  # Add label to the y-axis
    #plt.xticks(np.arange(1, len(hist['accuracy'])+1, 1))  # Set x-axis ticks to start from 1
    plt.legend(['train', 'validation'], loc='upper right')

    plt.show()

    # Accuracy Plot
    fig = plt.figure(figsize=(12,4))
    plt.title('Accuracy Plot')
    plt.plot(hist['accuracy'], '#07e9ed', marker='o')
    plt.plot(hist['val_accuracy'], '#0791ed',  marker='o')
    plt.ylim([0,1])

    plt.xlabel('Epoch')  # Add label to the x-axis
    plt.ylabel('Accuracy')  # Add label to the y-axis
    #plt.xticks(np.arange(1, len(hist['accuracy'])+1, 1))  # Set x-axis ticks to start from 1
    plt.legend(['train', 'validation'], loc='upper right')

    plt.show()
    # reset style
    style.use('default')
    #print(hist['accuracy'])

#all in one evaluating data, predicitring using test, and plotting all stuff
def classification_matrix_and_AUCcurve(model):

    # Evaluate the model on the test data
    evaluation_results = model.evaluate(testResnet)
    #Print the evaluation results
    print('Evaluation results (loss, accuracy, and AUC):', evaluation_results)

    print('\n-------------------------------')
    print('Predict results:\n')

    #classification report, predicting using test data
    predicted_vals = model.predict(testResnet)

    y_pred = np.where(predicted_vals >= 0.5,1 ,0) #change in the future for youdens j
    y_pred = y_pred.flatten()
    y_true = testResnet.classes  # Assuming test is a generator

    #y_true is from creating the dataframes, add zero_division=1 sa baba if gusto wala warning
    print(classification_report(y_true, y_pred))
    print("Accuracy of the Model:",accuracy_score(y_true, y_pred)*100,"%")

    #making confusion matrix thing
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Normal', 'Tuberculosis'],
                yticklabels=['Normal', 'Tuberculosis'])
    plt.xlabel('Predicted labels')
    plt.ylabel('True labels')
    plt.title('Confusion Matrix')
    plt.show()

    #for auc roc curve
    labels = ['tuberculosis']
    auc_roc_vals = []
    for i in range(len(labels)):
        try:
            gt = testResnet.labels[:]
            pred = predicted_vals[:, i]
            auc_roc = roc_auc_score(gt, pred)
            auc_roc_vals.append(auc_roc)
            fpr_rf, tpr_rf, _ = roc_curve(gt, pred)
            plt.figure(1, figsize=(10, 10))
            plt.plot([0, 1], [0, 1], 'k--')
            plt.plot(fpr_rf, tpr_rf,
                     label=labels[i] + " (" + str(round(auc_roc, 3)) + ")")
            plt.xlabel('False positive rate')
            plt.ylabel('True positive rate')
            plt.title('ROC curve')
            plt.legend(loc='best')
        except:
            print(
                f"Error in generating ROC curve for {labels[i]}. "
                f"Dataset lacks enough examples."
            )
    plt.show()


'''
#much simpler
history_frame = pd.DataFrame(resnet50_model.history)
history_frame.loc[:, ['loss', 'val_loss']].plot()
#history_frame.loc[:, ['binary_accuracy', 'val_binary_accuracy']].plot();
'''

print("Initializing plotting functions success")

# **Using the plotting functions to evaluate the model**

In [ ]:
plot_training_hist (historyResNet50)
classification_matrix_and_AUCcurve (model_ResNet50)

# **gradCAM functions**

In [ ]:
def get_img_array(img_path, size):
    img = tf.keras.preprocessing.image.load_img(img_path, target_size=size)
    array = tf.keras.preprocessing.image.img_to_array(img)
    # We add a dimension to transform our array into a "batch"
    # of size "size"
    array = np.expand_dims(array, axis=0)
    return array

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    # First, we create a model that maps the input image to the activations
    # of the last conv layer as well as the output predictions
    grad_model = tf.keras.models.Model(
        [model.inputs], [model.get_layer(last_conv_layer_name).output, model.output]
    )

    # Then, we compute the gradient of the top predicted class for our input image
    # with respect to the activations of the last conv layer
    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]
    # This is the gradient of the output neuron (top predicted or chosen)
    # with regard to the output feature map of the last conv layer
    grads = tape.gradient(class_channel, last_conv_layer_output)

    # This is a vector where each entry is the mean intensity of the gradient
    # over a specific feature map channel
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # We multiply each channel in the feature map array
    # by "how important this channel is" with regard to the top predicted class
    # then sum all the channels to obtain the heatmap class activation
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    # For visualization purpose, we will also normalize the heatmap between 0 & 1
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

'''
def save_and_display_gradcam(img_path, heatmap, cam_path="cam.jpg", alpha=0.4):
    # Load the original image
    img = tf.keras.preprocessing.image.load_img(img_path)
    img = tf.keras.preprocessing.image.img_to_array(img)

    # Rescale heatmap to a range 0-255
    heatmap = np.uint8(255 * heatmap)

    # Use jet colormap to colorize heatmap
    jet = cm.get_cmap("jet")

    # Use RGB values of the colormap
    jet_colors = jet(np.arange(256))[:, :3]
    jet_heatmap = jet_colors[heatmap]

    # Create an image with RGB colorized heatmap
    jet_heatmap = tf.keras.preprocessing.image.array_to_img(jet_heatmap)
    jet_heatmap = jet_heatmap.resize((img.shape[1], img.shape[0]))
    jet_heatmap = tf.keras.preprocessing.image.img_to_array(jet_heatmap)

    # Superimpose the heatmap on original image
    superimposed_img = jet_heatmap * alpha + img
    superimposed_img = tf.keras.preprocessing.image.array_to_img(superimposed_img)
    # Save the superimposed image
    superimposed_img.save(cam_path)

    # Display Grad CAM
    # display(Image(cam_path))

    return cam_path
'''

def superimpose_heatmap(original_img, heatmap, alpha=0.4):
    """
    Superimposes the heatmap on the original image.

    Args:
        original_img (numpy.ndarray): Original image.
        heatmap (numpy.ndarray): Heatmap generated by Grad-CAM.
        alpha (float): Weight of the heatmap when blending with the original image.

    Returns:
        numpy.ndarray: Superimposed image.
    """

    '''
    # Rescale heatmap to a range 0-255
    heatmap = np.uint8(255 * heatmap)

      # Resize heatmap to match original image dimensions
    heatmap_resized = cv2.resize(heatmap, (original_img.shape[1], original_img.shape[0]))

    # Apply colormap
    heatmap_color = cv2.applyColorMap(heatmap_resized, cv2.COLORMAP_JET)

    # Blend heatmap with the original image
    superimposed_img = cv2.addWeighted(original_img, alpha, heatmap_color, 1 - alpha, 0)

    plt.figure(figsize=(10, 5))
    plt.imshow(superimposed_img)
    plt.axis('off')
    plt.show()

    return superimposed_img
    '''
      # Load the original image
    img = keras.utils.load_img(original_img)
    img = keras.utils.img_to_array(img)

    # Rescale heatmap to a range 0-255
    heatmap = np.uint8(255 * heatmap)

    # Use jet colormap to colorize heatmap
    jet = mpl.colormaps["jet"]

    # Use RGB values of the colormap
    jet_colors = jet(np.arange(256))[:, :3]
    jet_heatmap = jet_colors[heatmap]

    # Create an image with RGB colorized heatmap
    jet_heatmap = keras.utils.array_to_img(jet_heatmap)
    jet_heatmap = jet_heatmap.resize((img.shape[1], img.shape[0]))
    jet_heatmap = keras.utils.img_to_array(jet_heatmap)

    # Superimpose the heatmap on original image
    superimposed_img = jet_heatmap * alpha + img
    superimposed_img = keras.utils.array_to_img(superimposed_img)

    # Save the superimposed image
    #superimposed_img.save(cam_path)

    # Display Grad CAM
    #display(Image(cam_path))

    return superimposed_img

print("Grad cam functions succesfully initialized")

# **using grad cam**

In [ ]:
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input
decode_predictions = tf.keras.applications.mobilenet_v2.decode_predictions

last_conv_layer_name = "conv5_block3_out"
img_size = (224,224, 3)

# Remove last layer's softmax
#pre_saved_model.layers[-1].activation = None

'''
# Display the part of the pictures used by the neural network to classify the pictures
fig, axes = plt.subplots(nrows=3, ncols=5, figsize=(15, 10),
                        subplot_kw={'xticks': [], 'yticks': []})

for i, ax in enumerate(axes.flat):
    img_path = test_df.Filepath.iloc[random_index[i]]
    img_array = preprocess_input(get_img_array(img_path, size=img_size))
    heatmap = make_gradcam_heatmap(img_array, model, last_conv_layer_name)
    cam_path = save_and_display_gradcam(img_path, heatmap)
    ax.imshow(plt.imread(cam_path))
    ax.set_title(f"True: {test_df.Label.iloc[random_index[i]]}\nPredicted: {pred[random_index[i]]}")
plt.tight_layout()
plt.show()
'''

# File path of the image (replace 'path_to_image.jpg' with the actual file path)
#normal = '/content/drive/MyDrive/THESIS/from-kaggle-dataset/Normal/Normal-1000.png'
# must investigate = image_path = '/content/drive/MyDrive/THESIS/from-kaggle-dataset/Tuberculosis/Tuberculosis-320.png'

#good example =  '/content/drive/MyDrive/THESIS/from-kaggle-dataset/Tuberculosis/Tuberculosis-10.png'
image_path = '/content/drive/MyDrive/THESIS/from-kaggle-dataset/Tuberculosis/Tuberculosis-10.png'

# Load the original image
original_img = cv2.imread(image_path)

# Get the image array
img_array = get_img_array(image_path, img_size)

#deactiviating last layer
model_ResNet50.layers[-1].activation = None

# Generate Grad-CAM heatmap (replace arguments with actual values)
heatmap = make_gradcam_heatmap(img_array, model_ResNet50, last_conv_layer_name)

# Display the superimposed image
superimpose_heatmap(image_path, heatmap)



'''
#to be fixed
single_img_generator_Gradcam = test_generator.flow(img_array, batch_size=1)

# Get the preprocessed image
preprocessed_img_Gradcam = next(single_img_generator_Gradcam)

predictionsGrad = model_Resnet_v2.predict(preprocessed_img_Gradcam)
# Print the predicted class label
if predictionsGrad[0][0] > 0.5:
    print(f'Score = {predictionsGrad[0][0]} | Tuberculosis')
else:
    print(f'Score = {predictionsGrad[0][0]} | Normal')

'''

'''
img = load_img('/kaggle/working/samples_random_online/normal/CHNCXR_0001_0.png', target_size=(HEIGHT, WIDTH))

# Create a new ImageDataGenerator for the single image
single_img_generator = test_generator.flow(np.array([img]), batch_size=1)

# Get the preprocessed image
preprocessed_img = next(single_img_generator)

predictions = resnet50_model.predict(preprocessed_img)
# Print the predicted class label
if predictions[0][0] > 0.5:
    print(f'Score = {predictions[0][0]} | Tuberculosis')
else:
    print(f'Score = {predictions[0][0]} | Normal')

'''

In [ ]:
img = load_img('/content/drive/MyDrive/THESIS/from-kaggle-dataset/Tuberculosis/Tuberculosis-700.png', target_size=(224, 224,3))

#activating again the classifier
model_ResNet50.layers[-1].activation = 'sigmoid'
'''
test_generator_single  = ImageDataGenerator(
                          rescale = 1./255,
                        )

# Create a new ImageDataGenerator for the single image
single_img_generator = test_generator_single.flow(np.array([img]), batch_size=1
                                                 )

# Get the preprocessed image
preprocessed_img = next(single_img_generator)

'''

# Convert image to array and preprocess it
img_array = img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
img_array = preprocess_input(img_array)


predictions = model_ResNet50.predict(img_array)
# Print the predicted class label
if predictions[0][0] > 0.5:
    print(f'Score = {predictions[0][0]} | Tuberculosis')
else:
    print(f'Score = {predictions[0][0]} | Normal')

#model_ResNet50.summary()

#USING LOADED MODEL,

In [ ]:
pre_saved_model = tf.keras.models.load_model('/content/drive/MyDrive/THESIS/ResNet50/Resnet50_model_v0-5/ResNet50_v5.keras')

pre_saved_model.summary()

''' #only if loading from weights
# Data Augmentation Step
augment = tf.keras.Sequential([
  layers.experimental.preprocessing.Resizing(224,224),
  layers.experimental.preprocessing.Rescaling(1./255),
  layers.experimental.preprocessing.RandomFlip("horizontal"),
  layers.experimental.preprocessing.RandomRotation(0.1),
  layers.experimental.preprocessing.RandomZoom(0.1),
  layers.experimental.preprocessing.RandomContrast(0.1),
])


base_load_model = ResNet50(
    input_shape=(224, 224, 3),
    include_top= False,
    weights= 'imagenet',
    pooling='max'
)

#base_model.load_weights('/kaggle/input/efficientnetb7-weights/efficientnetb7_notop.h5')
base_load_model.trainable = False

inputs = base__load_model.input


x = augment(inputs)
x = Dense(128, activation='relu')(base_load_model.output)
x = Dropout(0.45)(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.45)(x)

outputs = Dense(1, activation='sigmoid')(x)

loadedd_model_custom_weights = Model(inputs=inputs, outputs=outputs, name= 'resnet50_v3')

loadedd_model_custom_weights.load_weights('/content/drive/MyDrive/THESIS/ResNet50/resnet50modelv0-3.h5')

loadedd_model_custom_weights.compile(
    optimizer=Adam(0.00002), #change accordingly
    loss='binary_crossentropy',
    metrics=['accuracy', "AUC"]
)

loadedd_model_custom_weights.summary()

'''


In [ ]:
last_conv_layer_name = "conv5_block3_out"
img_size = (224,224, 3)

#good examples
tb = '/content/drive/MyDrive/THESIS/from-kaggle-dataset/Tuberculosis/Tuberculosis-10.png'
normal = '/content/drive/MyDrive/THESIS/from-kaggle-dataset/Normal/Normal-3000.png'

#bad examples
tb_bad = '/content/drive/MyDrive/THESIS/from-kaggle-dataset/Tuberculosis/Tuberculosis-330.png'
normal_bad  = '/content/drive/MyDrive/THESIS/from-kaggle-dataset/Normal/Normal-3500.png'

#what image to use
image_path = tb

# Load the original image
original_img = cv2.imread(image_path)

# Get the image array
img_array = get_img_array(image_path, img_size)


#Prediction results
#activating again the classifier
pre_saved_model.layers[-1].activation = 'sigmoid'

test_generator_single  = ImageDataGenerator(
                          preprocessing_function = preprocess_input
                        )

# Create a new ImageDataGenerator for the single image
single_img_generator = test_generator_single.flow(img_array, batch_size=1
                                                 )

# Get the preprocessed image
preprocessed_img = next(single_img_generator)

predictions_Presaved = pre_saved_model.predict(preprocessed_img)
# Print the predicted class label
if predictions_Presaved[0][0] > 0.5:
    print(f'Score = {predictions_Presaved[0][0]} | Tuberculosis')
else:
    print(f'Score = {predictions_Presaved[0][0]} | Normal')

#Using grad cam

# Remove last layer's softmax
pre_saved_model.layers[-1].activation = None

# Generate Grad-CAM heatmap (replace arguments with actual values)
heatmap = make_gradcam_heatmap(img_array, pre_saved_model, last_conv_layer_name)

# Display the superimposed image
superimposed_img = superimpose_heatmap(image_path, heatmap)

plt.figure(figsize=(10, 5))
plt.imshow(superimposed_img)
plt.axis('off')
plt.show()


print("tests done")

In [ ]:
#plot_training_hist (historyResNet50)
classification_matrix_and_AUCcurve (pre_saved_model)

In [ ]:
# multiple images example
last_conv_layer_name = "conv5_block3_out"
img_size = (224,224, 3)

import random

def generate_random_numbers(n, min_val=0, max_val=5):
    return [random.randint(min_val, max_val) for _ in range(n)]


random_index = np.random.randint(0, len(test_df), 5)

'''
def predict_random_images(dataframe, model, num_images=5):
    # Get random indices from the dataframe
    random_indices = generate_random_numbers(num_images, 0, len(dataframe)-1)

     # Iterate through random indices
    for i, idx in enumerate(random_indices):
        file_path = dataframe.loc[idx, 'FILE PATH']
        img = image.load_img(file_path, target_size=(224, 224))  # Assuming img_height and img_width are defined
        img_array = image.img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0)  # Expand dimensions to match model input
        predictions = model.predict(img_array)

        # Assuming model returns class probabilities, adjust as needed
        predicted_class = np.argmax(predictions, axis=1)[0]

        plt.subplot(1, num_images, i+1)
        plt.imshow(img)
        plt.title(f'Predicted class: {predicted_class}')
        plt.axis('off')

    plt.show()


predict_random_images(test_df, pre_saved_model)
'''

fig, axes = plt.subplots(nrows=1, ncols=5, figsize=(15, 10),
                        subplot_kw={'xticks': [], 'yticks': []})


def predict_random_images(dataframe, model, num_images=5, score_threshold=0.5):
  for i, ax in enumerate(axes.flat):
      img_path = dataframe['FILE PATH'].iloc[random_index[i]]
      #img_array = preprocess_input(get_img_array(img_path, size=img_size))

      #turn image into an array
      img_array = get_img_array(img_path, img_size)

      # Create a new ImageDataGenerator for the single image
      single_img_generator = test_generator_single.flow(img_array, batch_size=1
                                                      )
      #activate last layer
      model.layers[-1].activation = 'sigmoid'
      # Get the preprocessed image
      preprocessed_img = next(single_img_generator)

      predictions = model.predict(preprocessed_img)

      # Assuming model returns class probabilities, adjust as needed
      predicted_score = predictions[0][0]  # Probability of being TB positive

      # Adjusted predicted class label based on score threshold
      if predicted_score > 0.5:
              predicted_class = "tuberculosis"
      else:
              predicted_class = "normal"

      true_class = dataframe.label.iloc[random_index[i]]

      if true_class == predicted_class:
        color = "green"
      else:
        color = "red"

      #deactivate last layer again
      model.layers[-1].activation = None
      heatmap = make_gradcam_heatmap(img_array, pre_saved_model, last_conv_layer_name)

      superimposed_image = superimpose_heatmap (img_path, heatmap)

      plt.imshow(superimposed_image)


      ax.imshow(superimposed_image)
      ax.set_title(f"True Label: {true_class}\nPredicted: {predicted_class}", color=color)



  plt.show()
  plt.tight_layout()


predict_random_images(test_df, pre_saved_model)



EFFICIENTNET b3 v2 SETH

In [ ]:
pre_saved_model_efficientnet = tf.keras.models.load_model('/content/drive/MyDrive/THESIS/Copy-of-efficientNetB3model_v2.keras')

pre_saved_model_efficientnet.summary()

In [ ]:
img_size = (224,224, 3)

#good examples
tb = '/content/drive/MyDrive/THESIS/from-kaggle-dataset/Tuberculosis/Tuberculosis-10.png'
normal = '/content/drive/MyDrive/THESIS/from-kaggle-dataset/Normal/Normal-3000.png'

#bad examples
tb_bad = '/content/drive/MyDrive/THESIS/from-kaggle-dataset/Tuberculosis/Tuberculosis-330.png'
normal_bad  = '/content/drive/MyDrive/THESIS/from-kaggle-dataset/Normal/Normal-3500.png'

#what image to use
image_path = normal_bad

# Load the original image
original_img = cv2.imread(image_path)

# Get the image array
img_array = get_img_array(image_path, img_size)



test_generator_single  = ImageDataGenerator(
                          preprocessing_function = preprocess_input
                        )

# Create a new ImageDataGenerator for the single image
single_img_generator = test_generator_single.flow(img_array, batch_size=1
                                                 )

# Get the preprocessed image
preprocessed_img = next(single_img_generator)

predictions_Presaved = pre_saved_model_efficientnet.predict(preprocessed_img)
# Print the predicted class label
if predictions_Presaved[0][0] > 0.5:
    print(f'Score = {predictions_Presaved[0][0]} | Tuberculosis')
else:
    print(f'Score = {predictions_Presaved[0][0]} | Normal')


classification_matrix_and_AUCcurve (pre_saved_model_efficientnet)